In [3]:
#!/usr/bin/env python3
import requests
import subprocess
import datetime
from pathlib import Path

# ---------------- USER PATHS ----------------
ROOT = Path("/cluster/pixstor/stambaughm-lab/nasadem_conus")
CONUS_DIR = ROOT / "conus"/ "00_raw_nasadem"
ZIP_DIR = CONUS_DIR / "zips"
HGT_DIR = CONUS_DIR / "hgt"
LOG_DIR = ROOT / "logs"

ZIP_DIR.mkdir(parents=True, exist_ok=True)
HGT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

LOG_FILE = LOG_DIR / f"01b_download_nasadem_conus_cmr_{datetime.datetime.now():%Y%m%d_%H%M%S}.log"

def log(msg):
    ts = datetime.datetime.now().strftime("%F %T")
    line = f"[{ts}] {msg}"
    print(line)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

# ---------------- CMR SETTINGS ----------------
CMR_GRANULES_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"

PARAMS = {
    "short_name": "NASADEM_HGT",
    "bounding_box": "-125,25,-66,50",  # CONUS
    "page_size": 2000,
    "page_num": 1
}

HEADERS = {
    "Accept": "application/json"
}

def fetch_all_granules():
    granules = []
    page = 1

    while True:
        PARAMS["page_num"] = page
        log(f"Querying CMR page {page}")
        r = requests.get(CMR_GRANULES_URL, params=PARAMS, headers=HEADERS, timeout=60)
        r.raise_for_status()
        data = r.json()
        items = data["feed"]["entry"]

        if not items:
            break

        granules.extend(items)
        page += 1

    log(f"CMR returned {len(granules)} NASADEM granules")
    return granules

def extract_https_links(granules):
    urls = []
    for g in granules:
        for l in g.get("links", []):
            if (
                l.get("href", "").startswith("https://")
                and "lp-prod-protected" in l["href"]
                and l["href"].endswith(".zip")
            ):
                urls.append(l["href"])
    urls = sorted(set(urls))
    log(f"Extracted {len(urls)} HTTPS zip URLs")
    return urls

def curl_download(url, out_path):
    cmd = [
        "curl",
        "-L",              # follow redirects
        "-n",              # use ~/.netrc
        "-b", str(Path.home() / ".urs_cookies"),
        "-c", str(Path.home() / ".urs_cookies"),
        "--fail",          # fail on 4xx/5xx
        "--retry", "6",
        "--retry-delay", "2",
        "-o", str(out_path),
        url
    ]
    log(" ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        log(f"curl failed (code={p.returncode}) stderr:\n{p.stderr.strip()}")
    return p.returncode

def main():
    granules = fetch_all_granules()
    urls = extract_https_links(granules)

    for url in urls:
        fn = url.split("/")[-1]
        out_zip = ZIP_DIR / fn

        if out_zip.exists() and out_zip.stat().st_size > 0:
            continue

        rc = curl_download(url, out_zip)
        if rc != 0:
            if out_zip.exists() and out_zip.stat().st_size == 0:
                out_zip.unlink()
            continue

    # unzip idempotently
    for z in ZIP_DIR.glob("*.zip"):
        log(f"Unzipping {z.name}")
        subprocess.run(["unzip", "-n", "-d", str(HGT_DIR), str(z)], check=False)

    log("Download + unzip complete")
    log(f"ZIP_DIR={ZIP_DIR}")
    log(f"HGT_DIR={HGT_DIR}")

if __name__ == "__main__":
    main()

[2026-01-17 14:09:07] Querying CMR page 1
[2026-01-17 14:09:08] Querying CMR page 2
[2026-01-17 14:09:08] CMR returned 1250 NASADEM granules
[2026-01-17 14:09:08] Extracted 1250 HTTPS zip URLs
[2026-01-17 14:09:08] curl -L -n -b /home/mgvhy/.urs_cookies -c /home/mgvhy/.urs_cookies --fail --retry 6 --retry-delay 2 -o /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n24w075.zip https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/NASADEM_HGT.001/NASADEM_HGT_n24w075/NASADEM_HGT_n24w075.zip
[2026-01-17 14:09:10] curl -L -n -b /home/mgvhy/.urs_cookies -c /home/mgvhy/.urs_cookies --fail --retry 6 --retry-delay 2 -o /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n24w076.zip https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/NASADEM_HGT.001/NASADEM_HGT_n24w076/NASADEM_HGT_n24w076.zip
[2026-01-17 14:09:11] curl -L -n -b /home/mgvhy/.urs_cookies -c /home/mgvhy/.urs_cookies --fail --retry 6 --retry-delay 2 -o /cluster/pixstor/stambaugh

  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n25w125.zip or
        /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n25w125.zip.zip, and cannot find /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n25w125.zip.ZIP, period.


  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n40w120.hgt  
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n40w120.num  
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n40w120.swb  
[2026-01-17 14:40:14] Unzipping NASADEM_HGT_n27w103.zip
Archive:  /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n27w103.zip
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n27w103.hgt  
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n27w103.num  
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n27w103.swb  
[2026-01-17 14:40:15] Unzipping NASADEM_HGT_n32w098.zip
Archive:  /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/zips/NASADEM_HGT_n32w098.zip
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n32w098.hgt  
  inflating: /cluster/pixstor/stambaughm-lab/nasadem_conus/conus/hgt/n32w098.num  
  inflating: /cluster/pixstor/stambaughm-l